# NFT参照コードの使用例：コントラクトにapproveする例

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

動作確認に使用するユーザをロードします。

In [2]:
var users = [];
for(var i=0; i<=2; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct


ユーザ名の逆引き関数を準備します。

In [3]:
var userids = users.map(u => u.id);
function nameOfUserId(id){
    var i = userids.indexOf(id);
    if(i >= 0) return 'user'+i;
    return 'user_not_found';
}

## NFTコレクションのデプロイ

NFTコントラクトのサンプルコードを読み出します。

In [4]:
var { default: func } = await import('./nft100.mjs');
var m = /function.*\{([\s\S]*)\}/.exec(func.toString());
var code = m[1];

NFTコレクションの名称と略号を書き換えます。

In [5]:
var code = code.replace(/NFT_NAME = '.*'/, `NFT_NAME = 'sample NFT collection #2'`);
var code = code.replace(/NFT_SYMBOL = '.*'/, `NFT_SYMBOL = 'SMP2'`);

NFTコレクションの発行数を10とし、発行元をユーザ0に書き換えます。  
初期状態ではユーザ0が全トークンのオーナになります。

In [6]:
var code = code.replace(/TOTAL_SUPPLY = \d+/, `TOTAL_SUPPLY = 10`);
var code = code.replace(/INITIAL_OWNER = '.*'/, `INITIAL_OWNER = '${users[0].id}'`);

スマートコントラクトをデプロイします。  
変数nftidにNFTコントラクトのIDが格納されます。

In [7]:
var func = new Function(code);
var nftid = await deploySmartContract({func:'string', args:'any'}, func, { name: 'nft2' });

NFTコントラクトのアクセス範囲を、動作確認に使用するユーザに設定します。

In [8]:
await rpc.call(adminWallet, 'c1update', {id:nftid, prop:'accessible_to', value: userids});

{
  txno: 702254,
  txid: 'xLUzHED6VWmCpZiCVsEpK4CwgdLNsJVrHMvkfY3FjgCwMB',
  status: 'ok',
  value: null
}

## 例題用にトークンを配布

トークン１をユーザ１に転送し、トークン２をユーザ２に転送します。

In [9]:
var resp = await rpc.call(users[0].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[1].id,
      tokenId: 'token1'
    }
});
console.log(resp);

var resp = await rpc.call(users[0].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[0].id,
      to: users[2].id,
      tokenId: 'token2'
    }
});
console.log(resp);

{
  txno: 702255,
  txid: 'xNFtM6EkNX4ngiixy5CVEWX7h3wknMmeyPuSbHyXCFyWw',
  status: 'ok',
  value: null
}
{
  txno: 702256,
  txid: 'x5jr4Z7N9sUwStZHa7sY4hDAr2m9YBiid57sUG6Tkrpct',
  status: 'ok',
  value: null
}


念のためオーナを確認します。

In [10]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token1'}});
console.log('ownerOf(token1):', resp.value, nameOfUserId(resp.value));
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token2'}});
console.log('ownerOf(token2):', resp.value, nameOfUserId(resp.value));

ownerOf(token1): u28169743 user1
ownerOf(token2): u53371386 user2


## swapコントラクトをデプロイ

本例題用のコントラクトをデプロイします。  
ユーザ１とユーザ２の間でトークン１とトークン２を交換する処理を行うコントラクトです。  
これをswapコントラクトと呼ぶことにします。  
ユーティリティdeploySmartContractを使ってデプロイします。  
変数swap_cidにコントラクトのIDが返ります。

In [11]:
var swap_cid = await deploySmartContract({func:'string', args:'any'}, function nft_swap(func, args) {
    var nft = openContract('__nftid__');
    nft.call({func: 'transferFrom', args: { from: '__user1__', to: '__user2__', tokenId: 'token1' }});
    nft.call({func: 'transferFrom', args: { from: '__user2__', to: '__user1__', tokenId: 'token2' }});
}, {replacers: [
    [/__nftid__/g, nftid],
    [/__user1__/g, users[1].id],
    [/__user2__/g, users[2].id]
]});

作成したswapコントラクトがNFTコントラクトにアクセスできるように、accessible_toを設定します。

In [12]:
await rpc.call(adminWallet, 'c1update', {id:nftid, prop:'add accessible_to', value: [swap_cid]});

{
  txno: 702262,
  txid: 'xrVTZPCpdMPzPVF8r5wNA3dnKT94SbsZSFBct8ZiSQKeHB',
  status: 'ok',
  value: null
}

作成したコントラクトの内容を確認します。

In [13]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_contract', id: swap_cid });
console.log(resp.value);

{
  id: [ 'c096529695', 'nft_swap@c.TDSL.H011' ],
  frozen: null,
  name: 'nft_swap',
  domain: [ 'd93319890', '@c.TDSL.H011' ],
  description: '',
  argtypes: { func: 'string', args: 'any' },
  code: "    var nft = openContract('c076440605');    nft.call({func: 'transferFrom', args: { from: 'u28169743', to: 'u53371386', tokenId: 'token1' }});    nft.call({func: 'transferFrom', args: { from: 'u53371386', to: 'u28169743', tokenId: 'token2' }});",
  mask: {},
  maxsteps: 0,
  a_txno: 702259,
  c_txno: 702259,
  num_transactions: 0,
  last_active: 1753421641794,
  created_at: 1753421641794,
  accessible_to: [ [ 'u58281888', 'jupyter@c.TDSL.H011' ] ],
  callable_to: [ 'all' ],
  disclosed_to: [],
  groups: [],
  managed: true,
  accessible: true
}


## 実行例１

いきなりswapコントラクトを実行すると、エラーとなります。  
なぜなら、swapコントラクトにはトークンを転送する権限が無いからです。

In [14]:
await rpc.call(adminWallet, swap_cid);

{
  txno: 702263,
  txid: 'xuBmf4E9Po7QUUPdGTJuS2AzG7UTjUsuEmS4FJp9tc2KFB',
  status: 'aborted',
  value: 'thrown: caller not authorized for token1\n    at c096529695:1:46'
}

当然、トークンのオーナは変わっていません。

In [15]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token1'}});
console.log('ownerOf(token1):', resp.value, nameOfUserId(resp.value));
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token2'}});
console.log('ownerOf(token2):', resp.value, nameOfUserId(resp.value));

ownerOf(token1): u28169743 user1
ownerOf(token2): u53371386 user2


## 実行例２

トークン１の転送をapproveします。

In [16]:
var resp = await rpc.call(users[1].wallet, nftid, {func: 'approve', args: {approved: swap_cid, tokenId: 'token1'}});
console.log(resp);

{
  txno: 702267,
  txid: 'xAZu8DFrWCTZ9sAwdjc6BfFSuBTMmR4g3g7EDSwfdxoQRB',
  status: 'ok',
  value: null
}


念のため、getApprovedで確認します。

In [17]:
var resp = await rpc.call(users[1].wallet, nftid, {func: 'getApproved', args: {tokenId: 'token1'}});
console.log(resp);

{
  txno: 702268,
  txid: 'xCajJbf3NgerHxpTftnbxVTZKqe3FpiyAeRJoKjTyKM3u',
  status: 'ok',
  value: 'c096529695'
}


この状態で、swapコントラクトを実行すると、依然としてエラーとなります。  
なぜなら、トークン１は転送できても、swapコントラクトにトークン２を転送する権限が無いからです。  
エラーの起きる場所が前と変わっていることがわかります。

In [18]:
var resp = await rpc.call(adminWallet, swap_cid);
console.log(resp);

{
  txno: 702269,
  txid: 'x8enNQp3m3hUBXiQSyYP7GXhqY8BAq8vsMFLA4j6CHery',
  status: 'aborted',
  value: 'thrown: caller not authorized for token2\n    at c096529695:1:148'
}


この場合でも、いずれのトークンのオーナも変わっていません。  
トランザクションがエラーになることで、すべての変更が巻き戻されるからです。（トークン１の転送は元にもどります）

In [19]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token1'}});
console.log('ownerOf(token1):', resp.value, nameOfUserId(resp.value));
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token2'}});
console.log('ownerOf(token2):', resp.value, nameOfUserId(resp.value));

ownerOf(token1): u28169743 user1
ownerOf(token2): u53371386 user2


## 実行例３

さらに、トークン２の転送をapproveします。

In [20]:
var resp = await rpc.call(users[2].wallet, nftid, {func: 'approve', args: {approved: swap_cid, tokenId: 'token2'}});
console.log(resp);

{
  txno: 702274,
  txid: 'xTGthkDaQFttUcmJbTPVn6E8NuXHFJEScYgTRCvPGog5t',
  status: 'ok',
  value: null
}


念のため、getApprovedで確認します。

In [21]:
var resp = await rpc.call(users[1].wallet, nftid, {func: 'getApproved', args: {tokenId: 'token2'}});
console.log(resp);

{
  txno: 702275,
  txid: 'xC8opqh9aGLzcbKXz8P9HJcknCi8VwU4NgSugn5UPAYGo',
  status: 'ok',
  value: 'c096529695'
}


この状態で、swapコントラクトを実行すると、実行は成功します。

In [22]:
var resp = await rpc.call(adminWallet, swap_cid);
console.log(resp);

{
  txno: 702276,
  txid: 'xSsMebjeSKUufAnS8AWZkfWVkv46UQr3XoYnqUAbxakQ4',
  status: 'ok',
  value: null
}


トークンのオーナが入れ替わっていることを確認します。

In [23]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token1'}});
console.log('ownerOf(token1):', resp.value, nameOfUserId(resp.value));
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: 'token2'}});
console.log('ownerOf(token2):', resp.value, nameOfUserId(resp.value));

ownerOf(token1): u53371386 user2
ownerOf(token2): u28169743 user1


このような仕組みで、アトミックにNFTの交換を実現することができました。